# Supplemental Table S3 — N30 · U35 H-bond lifetimes

Per-clone lifetimes and half-lives; mean ± SEM across clones that formed ≥1 complete H-bond event.

Data: `../data/hbond-atom/`

In [1]:
from pathlib import Path
FIG_DIR = Path('figures'); FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR = Path('tables'); TABLE_DIR.mkdir(exist_ok=True)

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import re
from typing import Dict, List, Tuple, Optional
from collections import defaultdict
from scipy import stats


## Configuration

Same parameters as `hbonds_lifetime.ipynb`.

In [3]:
# Data directory
DATA_DIR = Path('../data/hbond-atom')

# Analysis parameters
ANALYSIS_START_FRAME = 13000   # Start of production analysis window
ANALYSIS_END_FRAME = 23000     # End of production analysis window
GAP_TOLERANCE = 0              # Frames to bridge gaps (0 = no bridging)
TIME_PER_FRAME_NS = 0.1        # 100 ps = 0.1 ns per frame

# System definitions
SYSTEMS = [
    '20251112_eIF5wt_AUG', '20251112_eIF5wt_ACG', '20251112_eIF5wt_CUG', '20251112_eIF5wt_GUG',
    '20251112_eIF5wt_UUG', '20251112_eIF5wt_AUC',
    '20251112_eIF5_N30K_AUG', '20251112_eIF5_N30Q_AUG', '20251112_eIF5_G29A_G31A_AUG'
]

SYSTEM_LABELS = {
    '20251112_eIF5wt_AUG': 'wt/AUG',
    '20251112_eIF5wt_ACG': 'wt/ACG',
    '20251112_eIF5wt_CUG': 'wt/CUG',
    '20251112_eIF5wt_UUG': 'wt/UUG',
    '20251112_eIF5wt_AUC': 'wt/AUC',
    '20251112_eIF5wt_GUG': 'wt/GUG',
    '20251112_eIF5_N30K_AUG': 'N30K/AUG',
    '20251112_eIF5_N30Q_AUG': 'N30Q/AUG',
    '20251112_eIF5_G29A_G31A_AUG': 'G29A_G31A/AUG'
}

# Atom to analyze
SELECTED_ATOM = 'U_1912@O2'


## Data Loading and Event Extraction

Same per-frame extraction machinery as `hbonds_lifetime.ipynb`.

In [4]:
def parse_filename(filename: str) -> Optional[Dict[str, str]]:
    """
    Parse filename to extract system information.

    Example: 20251112_48S_eIF5wt_AUG_clone0-HB_N30-matrix_atom.dat

    Returns:
        Dictionary with date, variant, codon, clone, system keys
    """
    pattern = r'(\d{8})_48S_(eIF5.*?)_(AUG|ACG|CUG|UUG|AUC|GUG)_clone(\d+)-HB_N30-matrix_atom\.dat'
    match = re.match(pattern, filename)

    if match:
        return {
            'date': match.group(1),
            'variant': match.group(2),
            'codon': match.group(3),
            'clone': match.group(4),
            'system': f"{match.group(1)}_{match.group(2)}_{match.group(3)}"
        }
    return None


In [5]:
def load_atom_matrix(filepath: Path) -> Tuple[np.ndarray, List[str], Dict[str, str]]:
    """
    Load an atom-level H-bond matrix file.

    Returns:
        matrix: 2D array (frames x atoms) of H-bond counts
        atom_names: List of atom column names
        info: Parsed filename information
    """
    info = parse_filename(filepath.name)
    if info is None:
        raise ValueError(f"Could not parse filename: {filepath.name}")

    with open(filepath, 'r') as f:
        header = f.readline().strip()

    # Split by whitespace (tabs or spaces), skip '#Frame'
    atom_names = header.split()[1:]

    data = np.loadtxt(filepath, skiprows=1)
    matrix = data[:, 1:].astype(int)  # Skip frame column

    return matrix, atom_names, info


In [6]:
def extract_binding_events(
    time_series: np.ndarray,
    gap_tolerance: int = 0
) -> List[Tuple[int, int, bool, bool]]:
    """
    Extract continuous binding events from binary time series.

    A binding event is a contiguous stretch where H-bond count >= 1.

    Returns:
        List of (start_frame, end_frame, left_censored, right_censored)
    """
    events: List[Tuple[int, int, bool, bool]] = []
    n = len(time_series)
    i = 0

    while i < n:
        if time_series[i] >= 1:  # Bond exists
            start = i
            while i < n and time_series[i] >= 1:
                i += 1
            end = i - 1  # Last frame with bond

            left_censored = (start == 0)
            right_censored = (end == n - 1)
            events.append((start, end, left_censored, right_censored))
        else:
            i += 1

    return events


def filter_complete_events(
    events: List[Tuple[int, int, bool, bool]]
) -> List[Tuple[int, int]]:
    """Remove censored events, keeping only complete observations."""
    complete: List[Tuple[int, int]] = []
    for start, end, left_cens, right_cens in events:
        if not left_cens and not right_cens:
            complete.append((start, end))
    return complete


def calculate_lifetimes(
    complete_events: List[Tuple[int, int]]
) -> List[int]:
    """Lifetime = end_frame - start_frame + 1 (inclusive count), in frames."""
    return [end - start + 1 for start, end in complete_events]


In [7]:
def calculate_survival_function(
    lifetimes: List[int],
    time_per_frame_ns: float = TIME_PER_FRAME_NS
) -> Tuple[np.ndarray, np.ndarray]:
    """Empirical survival function S(t) = P(lifetime > t), via complement of the ECDF."""
    if len(lifetimes) == 0:
        return np.array([0.0]), np.array([1.0])

    sorted_lifetimes = np.sort(lifetimes)
    n = len(sorted_lifetimes)
    unique_lifetimes = np.unique(sorted_lifetimes)

    time_points = [0]
    survival = [1.0]

    for t in unique_lifetimes:
        s_t = np.sum(sorted_lifetimes > t) / n
        time_points.append(t)
        survival.append(s_t)

    time_points_ns = np.array(time_points) * time_per_frame_ns
    survival_probability = np.array(survival)

    return time_points_ns, survival_probability


def find_median_lifetime(
    time_points: np.ndarray,
    survival: np.ndarray
) -> Optional[float]:
    """Half-life = time at which survival crosses 0.5, via linear interpolation."""
    if len(time_points) < 2:
        return None

    for i in range(len(survival) - 1):
        if survival[i] >= 0.5 and survival[i + 1] < 0.5:
            t1, t2 = time_points[i], time_points[i + 1]
            s1, s2 = survival[i], survival[i + 1]
            return t1 + (0.5 - s1) * (t2 - t1) / (s2 - s1)

    if survival[0] < 0.5:
        return 0.0

    return None  # Survival never drops below 0.5


def half_life_from_lifetimes(
    lifetimes: List[int],
    time_per_frame_ns: float = TIME_PER_FRAME_NS
) -> Optional[float]:
    """Half-life of an H-bond: time at which S(t) = 0.5, in ns."""
    time_pts, surv = calculate_survival_function(lifetimes, time_per_frame_ns)
    return find_median_lifetime(time_pts, surv)


## Load Files, Grouped by System

In [8]:
# Load all atom matrix files and organize by system
all_files = sorted(DATA_DIR.glob('*-matrix_atom.dat'))
print(f"Found {len(all_files)} atom matrix files")

systems_files: Dict[str, List[Path]] = {}
for filepath in all_files:
    info = parse_filename(filepath.name)
    if info is None:
        continue

    system_key = info['system']
    if system_key not in systems_files:
        systems_files[system_key] = []
    systems_files[system_key].append(filepath)

print(f"Organized into {len(systems_files)} unique systems")
for system_key in sorted(systems_files.keys()):
    print(f"  {system_key}: {len(systems_files[system_key])} clones")


Found 72 atom matrix files
Organized into 9 unique systems
  20251112_eIF5_G29A_G31A_AUG: 8 clones
  20251112_eIF5_N30K_AUG: 8 clones
  20251112_eIF5_N30Q_AUG: 8 clones
  20251112_eIF5wt_ACG: 8 clones
  20251112_eIF5wt_AUC: 8 clones
  20251112_eIF5wt_AUG: 8 clones
  20251112_eIF5wt_CUG: 8 clones
  20251112_eIF5wt_GUG: 8 clones
  20251112_eIF5wt_UUG: 8 clones


## Per-Clone Lifetime Extraction

Unlike `hbonds_lifetime.ipynb`'s `process_all_lifetimes()`, events are kept
**separated by clone** instead of pooled: `Dict[system][atom][clone] -> lifetimes`.

**Caveat:** cpptraj's `series uuseries` only emits a column for an atom if it
formed at least one H-bond during that trajectory. A clone where the selected
atom never bonded has no column for it at all in its matrix file -- a genuine
zero-event observation, not missing data. `process_lifetimes_by_clone()`
explicitly fills these in as `[]` per clone so they are not silently dropped
from `N_Clones`/the cross-clone mean and SEM.

In [9]:
def process_lifetimes_by_clone(
    systems_files: Dict[str, List[Path]],
    start_frame: int = ANALYSIS_START_FRAME,
    end_frame: int = ANALYSIS_END_FRAME,
    gap_tolerance: int = GAP_TOLERANCE
) -> Dict[str, Dict[str, Dict[str, List[int]]]]:
    """
    Process all systems and extract lifetimes for each atom, kept per clone.

    Important: cpptraj's `series uuseries` only writes a column for an atom if
    it formed at least one H-bond during that trajectory. A clone where an
    atom never bonded therefore has NO column for it in its matrix file at
    all -- that's a real "zero events" observation, not missing data. Every
    clone present in `systems_files[system]` is represented for every atom
    seen in ANY of that system's clones, defaulting to an empty lifetime list
    ([]) when the atom's column is absent from a particular clone's file, so
    such clones don't silently disappear from downstream statistics.

    Args:
        systems_files: Dict mapping system_key -> list of file paths
        start_frame: Start of analysis window
        end_frame: End of analysis window
        gap_tolerance: Frames to bridge gaps

    Returns:
        Dict[system][atom][clone] -> list of lifetimes (frames); [] means
        either the atom's column was absent from that clone's file, or it was
        present but produced no complete (uncensored) binding events.
    """
    lifetimes_by_clone: Dict[str, Dict[str, Dict[str, List[int]]]] = {}

    for system_key, file_paths in sorted(systems_files.items()):
        clone_ids: List[str] = []
        per_clone_lifetimes: Dict[str, Dict[str, List[int]]] = {}
        atoms_seen: set = set()

        for filepath in file_paths:
            matrix, atom_names, info = load_atom_matrix(filepath)
            clone_id = info['clone']
            clone_ids.append(clone_id)
            atoms_seen.update(atom_names)

            matrix_windowed = matrix[start_frame:end_frame, :]

            clone_atom_lifetimes: Dict[str, List[int]] = {}
            for i, atom_name in enumerate(atom_names):
                time_series = matrix_windowed[:, i]
                events = extract_binding_events(time_series, gap_tolerance)
                complete = filter_complete_events(events)
                clone_atom_lifetimes[atom_name] = calculate_lifetimes(complete)

            per_clone_lifetimes[clone_id] = clone_atom_lifetimes

        # Re-key as [atom][clone], filling in [] for any clone whose file had
        # no column for that atom (bond never formed in that clone).
        system_data: Dict[str, Dict[str, List[int]]] = {}
        for atom_name in atoms_seen:
            system_data[atom_name] = {
                clone_id: per_clone_lifetimes[clone_id].get(atom_name, [])
                for clone_id in clone_ids
            }

        lifetimes_by_clone[system_key] = system_data

    return lifetimes_by_clone


print(f"Analysis parameters:")
print(f"  Frame range: {ANALYSIS_START_FRAME} - {ANALYSIS_END_FRAME}")
print(f"  Gap tolerance: {GAP_TOLERANCE} frames")
print(f"  Time per frame: {TIME_PER_FRAME_NS} ns\n")

lifetimes_by_clone = process_lifetimes_by_clone(systems_files)
print("Done.")


Analysis parameters:
  Frame range: 13000 - 23000
  Gap tolerance: 0 frames
  Time per frame: 0.1 ns



Done.


## Per-Clone Statistics

One row per (system, clone): number of events, mean lifetime, and half-life —
each computed from that single clone's binding events only. This is the
intermediate table the cross-clone stochastic error is derived from.

In [10]:
def calculate_clone_level_stats(
    lifetimes_by_clone: Dict[str, Dict[str, Dict[str, List[int]]]],
    atom: str,
    systems: List[str],
    time_per_frame_ns: float = TIME_PER_FRAME_NS
) -> pd.DataFrame:
    """
    Per-clone dynamics for a single atom, across the requested systems.

    Args:
        lifetimes_by_clone: Dict[system][atom][clone] -> list of lifetimes (frames)
        atom: Atom name to extract (e.g. 'U_1912@O2')
        systems: Systems to include
        time_per_frame_ns: Conversion factor (ns per frame)

    Returns:
        DataFrame with one row per (system, clone): System, Clone, N_Events,
        Mean_Lifetime_ns, HalfLife_ns.
    """
    rows: List[Dict] = []

    for system_key in systems:
        atom_data = lifetimes_by_clone.get(system_key, {}).get(atom, {})

        for clone_id, lifetimes in sorted(atom_data.items()):
            n_events = len(lifetimes)
            if n_events == 0:
                mean_lifetime = np.nan
                half_life = np.nan
            else:
                lifetimes_ns = np.asarray(lifetimes) * time_per_frame_ns
                mean_lifetime = float(np.mean(lifetimes_ns))
                hl = half_life_from_lifetimes(lifetimes, time_per_frame_ns)
                half_life = hl if hl is not None else np.nan

            rows.append({
                'System': SYSTEM_LABELS.get(system_key, system_key),
                'System_Key': system_key,
                'Clone': clone_id,
                'N_Events': n_events,
                'Mean_Lifetime_ns': mean_lifetime,
                'HalfLife_ns': half_life,
            })

    return pd.DataFrame(rows)


clone_stats_df = calculate_clone_level_stats(lifetimes_by_clone, SELECTED_ATOM, SYSTEMS)
print(f"Per-clone stats for {SELECTED_ATOM}: {len(clone_stats_df)} (system, clone) rows")
display(clone_stats_df)


Per-clone stats for U_1912@O2: 72 (system, clone) rows


,System,System_Key,Clone,N_Events,Mean_Lifetime_ns,HalfLife_ns
0,wt/AUG,20251112_eIF5wt_AUG,0,602,0.723422,0.285526
1,wt/AUG,20251112_eIF5wt_AUG,1,409,0.483374,0.209375
2,wt/AUG,20251112_eIF5wt_AUG,2,203,0.882266,0.380769
3,wt/AUG,20251112_eIF5wt_AUG,3,1399,0.371122,0.126589
4,wt/AUG,20251112_eIF5wt_AUG,4,425,0.960235,0.170192
...,...,...,...,...,...,...
67,G29A_G31A/AUG,20251112_eIF5_G29A_G31A_AUG,3,99,0.394949,0.131250
68,G29A_G31A/AUG,20251112_eIF5_G29A_G31A_AUG,4,254,0.192520,0.084667
69,G29A_G31A/AUG,20251112_eIF5_G29A_G31A_AUG,5,1068,0.292790,0.135294
70,G29A_G31A/AUG,20251112_eIF5_G29A_G31A_AUG,6,72,0.215278,0.083721


In [11]:
clone_stats_df[clone_stats_df['System'] == 'wt/AUG']

,System,System_Key,Clone,N_Events,Mean_Lifetime_ns,HalfLife_ns
0,wt/AUG,20251112_eIF5wt_AUG,0,602,0.723422,0.285526
1,wt/AUG,20251112_eIF5wt_AUG,1,409,0.483374,0.209375
2,wt/AUG,20251112_eIF5wt_AUG,2,203,0.882266,0.380769
3,wt/AUG,20251112_eIF5wt_AUG,3,1399,0.371122,0.126589
4,wt/AUG,20251112_eIF5wt_AUG,4,425,0.960235,0.170192
5,wt/AUG,20251112_eIF5wt_AUG,5,736,0.724728,0.231944
6,wt/AUG,20251112_eIF5wt_AUG,6,644,0.440528,0.215663
7,wt/AUG,20251112_eIF5wt_AUG,7,799,0.553442,0.199020


## Per-System Dynamics Table (Cross-Clone SEM)

Aggregating the per-clone table above:

- **`N_Clones`** — total clones run for the system (e.g. 8), regardless of whether the bond ever formed.
- **`N_Clones_With_Bond`** — clones with >=1 complete binding event; this is the *n* the mean/SEM/SD below are computed from, since a clone where the bond never formed has no defined lifetime/half-life.
- **`N_Events`** — total binding events for the system (**summed** across all clones, including zero from clones with no bond).
- **`HalfLife_Mean_ns`**, **`MeanLifetime_Mean_ns`** — mean of the per-clone half-life / mean-lifetime values (over `N_Clones_With_Bond` clones).
- **`*_SEM_ns`** — `scipy.stats.sem` across those clones: the stochastic error on the reported mean, i.e. how much this estimate would be expected to vary if the simulation were rerun. This is the number to use for comparing systems.
- **`*_SD_ns`** — standard deviation across those clones: how much the clones themselves differ from one another (descriptive, not an error bar on the mean).

If `N_Clones_With_Bond < N_Clones` for a system, that means the bond failed to form at all in one or more replicates — worth checking/reporting separately, since it changes the interpretation from "how long does the bond last" to "does the bond form reliably in the first place."

In [12]:
def calculate_atom_dynamics_table(
    clone_stats_df: pd.DataFrame,
    systems: List[str],
    time_per_frame_ns: float = TIME_PER_FRAME_NS
) -> pd.DataFrame:
    """
    Per-system H-bond dynamics, aggregating per-clone statistics with cross-clone SEM.

    Args:
        clone_stats_df: Output of calculate_clone_level_stats()
        systems: Systems to include, in display order
        time_per_frame_ns: Conversion factor (unused directly; kept for signature symmetry)

    Returns:
        DataFrame with one row per system: N_Clones (all clones for the
        system), N_Clones_With_Bond (clones with >=1 complete event, i.e. the
        n used for the mean/SEM/SD below), N_Events (summed across all
        clones), HalfLife_Mean_ns/SEM/SD, MeanLifetime_Mean_ns/SEM/SD.
    """
    rows: List[Dict] = []

    for system_key in systems:
        sub = clone_stats_df[clone_stats_df['System_Key'] == system_key]
        if len(sub) == 0:
            continue

        n_clones_total = int(sub['Clone'].nunique())
        n_events_total = int(sub['N_Events'].sum())

        half_lives = sub['HalfLife_ns'].dropna().to_numpy()
        mean_lifetimes = sub['Mean_Lifetime_ns'].dropna().to_numpy()
        n_clones_with_bond = int((sub['N_Events'] > 0).sum())

        def mean_sem_sd(values: np.ndarray) -> Tuple[float, float, float]:
            if len(values) == 0:
                return np.nan, np.nan, np.nan
            if len(values) == 1:
                return float(values[0]), np.nan, np.nan
            return (
                float(np.mean(values)),
                float(stats.sem(values, ddof=1)),
                float(np.std(values, ddof=1)),
            )

        hl_mean, hl_sem, hl_sd = mean_sem_sd(half_lives)
        ml_mean, ml_sem, ml_sd = mean_sem_sd(mean_lifetimes)

        rows.append({
            'System': SYSTEM_LABELS.get(system_key, system_key),
            'System_Key': system_key,
            'N_Clones': n_clones_total,
            'N_Clones_With_Bond': n_clones_with_bond,
            'N_Events': n_events_total,
            'HalfLife_Mean_ns': hl_mean,
            'HalfLife_SEM_ns': hl_sem,
            'HalfLife_SD_ns': hl_sd,
            'MeanLifetime_Mean_ns': ml_mean,
            'MeanLifetime_SEM_ns': ml_sem,
            'MeanLifetime_SD_ns': ml_sd,
        })

    df = pd.DataFrame(rows)
    return df


dynamics_df = calculate_atom_dynamics_table(clone_stats_df, SYSTEMS)
print(f"H-bond dynamics for {SELECTED_ATOM}, per system (mean +/- SEM across clones):\n")
print("Note: HalfLife/MeanLifetime mean/SEM/SD are computed only over clones")
print("with N_Events > 0 (N_Clones_With_Bond); clones with zero events have no")
print("defined lifetime and are excluded from that average, but still count")
print("toward N_Clones and contribute 0 to N_Events.\n")
display(dynamics_df)


H-bond dynamics for U_1912@O2, per system (mean +/- SEM across clones):

Note: HalfLife/MeanLifetime mean/SEM/SD are computed only over clones
with N_Events > 0 (N_Clones_With_Bond); clones with zero events have no
defined lifetime and are excluded from that average, but still count
toward N_Clones and contribute 0 to N_Events.



,System,System_Key,N_Clones,N_Clones_With_Bond,N_Events,HalfLife_Mean_ns,HalfLife_SEM_ns,HalfLife_SD_ns,MeanLifetime_Mean_ns,MeanLifetime_SEM_ns,MeanLifetime_SD_ns
0,wt/AUG,20251112_eIF5wt_AUG,8,8,5217,0.227385,0.027264,0.077113,0.642390,0.075527,0.213623
1,wt/ACG,20251112_eIF5wt_ACG,8,8,2191,0.243041,0.077621,0.219544,0.572466,0.161635,0.457174
2,wt/CUG,20251112_eIF5wt_CUG,8,8,3613,0.086642,0.007112,0.020115,0.219795,0.026173,0.074028
3,wt/GUG,20251112_eIF5wt_GUG,8,5,129,0.108696,0.035303,0.078941,0.250104,0.096055,0.214786
4,wt/UUG,20251112_eIF5wt_UUG,8,8,5897,0.119767,0.019698,0.055714,0.310686,0.055728,0.157623
5,wt/AUC,20251112_eIF5wt_AUC,8,8,1820,0.223921,0.022908,0.064794,0.599975,0.054386,0.153826
6,N30K/AUG,20251112_eIF5_N30K_AUG,8,4,1490,0.088425,0.004841,0.009682,0.208640,0.009073,0.018146
7,N30Q/AUG,20251112_eIF5_N30Q_AUG,8,7,1064,0.224058,0.022167,0.058648,0.541011,0.040171,0.106281
8,G29A_G31A/AUG,20251112_eIF5_G29A_G31A_AUG,8,8,2542,0.142512,0.025392,0.071820,0.431902,0.109676,0.310210


## Export

In [13]:
table = dynamics_df[['System', 'N_Clones_With_Bond', 'N_Events',
                    'MeanLifetime_Mean_ns', 'MeanLifetime_SEM_ns',
                    'HalfLife_Mean_ns', 'HalfLife_SEM_ns']].rename(columns={
    'N_Clones_With_Bond': 'Trajectories with ≥1 H-bond',
    'N_Events': 'Total H-bonds',
    'MeanLifetime_Mean_ns': 'Lifetime mean (ns)',
    'MeanLifetime_SEM_ns': 'Lifetime SEM (ns)',
    'HalfLife_Mean_ns': 'Half-life mean (ns)',
    'HalfLife_SEM_ns': 'Half-life SEM (ns)',
})
table.to_csv(TABLE_DIR / 'TableS3_N30_U35_hbond_lifetime.csv', index=False)
table.round(2)

,System,Trajectories with ≥1 H-bond,Total H-bonds,Lifetime mean (ns),Lifetime SEM (ns),Half-life mean (ns),Half-life SEM (ns)
0,wt/AUG,8,5217,0.64,0.08,0.23,0.03
1,wt/ACG,8,2191,0.57,0.16,0.24,0.08
2,wt/CUG,8,3613,0.22,0.03,0.09,0.01
3,wt/GUG,5,129,0.25,0.10,0.11,0.04
4,wt/UUG,8,5897,0.31,0.06,0.12,0.02
5,wt/AUC,8,1820,0.60,0.05,0.22,0.02
6,N30K/AUG,4,1490,0.21,0.01,0.09,0.00
7,N30Q/AUG,7,1064,0.54,0.04,0.22,0.02
8,G29A_G31A/AUG,8,2542,0.43,0.11,0.14,0.03
